# Groundwater Production Dossier (GPD)
A Groundwater Production Dossier contains the production figures of a groundwater use system.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import brodata

In [ ]:
gpd = brodata.gpd.GroundwaterProductionDossier.from_bro_id("GPD000000012107")

There are multiple reports in this Groundwater Production Dossier, where each report contains multiple volume-registrations. The metadata of the reports are contained in the attribute `report`, and the volumes are contained in the attribute `volumeSeries`. These two pandas DataFrames are linked to eachother by the column `reportId`.

In [ ]:
gpd.report

Show the volume-series

In [ ]:
gpd.volumeSeries

Plot the volume-series, and sperate the volumes by the injected or extracted attribute, and, if injected, by the temperature-description.

In [ ]:
f, ax = plt.subplots(figsize=(10, 6))
ax.axhline(0, color="black", lw=0.5)
for waterInOut in gpd.volumeSeries["waterInOut"].unique():
    maskInOut = gpd.volumeSeries["waterInOut"] == waterInOut
    for temperatureIn in gpd.volumeSeries.loc[maskInOut, "temperatureIn"].unique():
        if pd.isna(temperatureIn):
            maskTemperatureIn = gpd.volumeSeries["temperatureIn"].isna()
            label = f"{waterInOut}"
        else:
            maskTemperatureIn = gpd.volumeSeries["temperatureIn"] == temperatureIn
            label = f"{waterInOut} - {temperatureIn}"
        mask = maskInOut & maskTemperatureIn
        df = gpd.volumeSeries.loc[mask]

        # sort by the column `beginDate`
        df = df.sort_values("beginDate")

        # make a matplotlib plot of the volume series, with a constant value between beginDate and endDate
        x = df.loc[:, ["beginDate", "endDate"]].values.ravel()

        y = np.vstack((df["volume"].values, df["volume"].values)).transpose().ravel()
        if waterInOut == "onttrokken":
            y = -y
        
        ax.plot(x, y, label=label)
ax.set_xlim(gpd.volumeSeries["beginDate"].min(), gpd.volumeSeries["endDate"].max())
ax.legend();

Show the rest of the contents of the Groundwater Production Dossier.

In [ ]:
gpd_data = gpd.to_dict()
gpd_data.pop("report")
gpd_data.pop("volumeSeries")
gpd_data